# GBIF cube to ISEA3H hexagons for Hudson Bay Lowlands

This notebook goes from a GBIF query to a hexagon layer in a single run:

1. Builds the **SQL query** (same filters, footprint polygons and ISEA3H grid as the original download).

2. Requests the **GBIF SQL download through the API** and waits for it to finish (or reuses a download key you already have, for example the link for this dowload would be:  https://www.gbif.org/occurrence/download/0007835-260928105237408 and the DOWNLOAD_KEY would be "0007835-260928105237408", but all of this is also in the code in case the dowload expires).

3. Downloads and unzips the cube (a CSV with `isea3hcellcode` cell codes, no geometry).

4. Rebuilds the hexagons with the same projection GBIF uses (Snyder equal-area projection on an icosahedron, a Python port of the `geogrid` library), adds per-cell attributes and the species list, keeps the hexagons that touch the HJBL footprint and writes a single GeoJSON file.

**Before you run it**
- `data-processing/.env` must define `GBIF_USER`, `GBIF_PWD` (your GBIF account, no token needed) and `GBIF_EMAIL` in .env.

- The footprint must exist: run `01_get_footprint.ipynb` first.

- Needs `requests` and `python-dotenv` in the environment (`uv add requests python-dotenv` if they are missing).

**How to use it:** edit section 1 and run the cells from top to bottom. Sections 2 to 7 only define constants and functions. The actual work starts in section 8.

> **Note:** Core functions, GBIF data extraction, ISEA3H cell decoding, and hexagon generation are located in `src/data_processing/gbif.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
from pathlib import Path

from dotenv import load_dotenv

from data_processing.gbif import (
    aggregate_cells,
    build_hexagon_layer,
    build_sql,
    download_cube,
    filter_by_footprint,
    load_cube,
    request_download,
    save_geojson,
)

# Show INFO messages in the notebook output
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s", force=True)
LOGGER = logging.getLogger("gbif_hjbl_hexagons")
for noisy in ("pyogrio", "fiona", "urllib3"):    # silence chatter from the I/O and HTTP libraries
    logging.getLogger(noisy).setLevel(logging.WARNING)

load_dotenv(Path("../.env"))                      # data-processing/.env (GBIF_USER, GBIF_PWD, GBIF_EMAIL)

## 1. Settings
This is the only part you need to edit, depending of the download you want to use.

In [ ]:
# --- GBIF download ---
DOWNLOAD_KEY = "0007835-260928105237408"      # None = request a NEW download. Or reuse one, e.g. "0007835-260928105237408"
ISEA3H_LEVEL = 16         # ISEA3H resolution (16 -> hexagons of about 1.19 km2)
POLL_SECONDS = 30         # seconds between status checks while GBIF prepares the download
VALIDATE_FIRST = True     # ask GBIF to validate the SQL before submitting it

# --- Paths (relative to data-processing/notebooks) ---
FOOTPRINT = Path("../data/processed/vectors/HJBL_footprint.geojson")  # made by 01_get_footprint.ipynb; None = keep every hexagon
RAW_DIR = Path("../data/raw/gbif")                # downloaded GBIF files (one subfolder per download key)
OUT_DIR = Path("../data/processed/vectors")       # output folder
OUT_NAME = "gbif_richness"                      # base name of the output files (_L<level> is appended)
EQUAL_AREA_CRS = "ESRI:102001"                    # Canada Albers Equal Area Conic, used only to compute area_km2

# If True, adds the first_year and last_year columns
INCLUDE_YEARS = True

REQUIRED_COLUMNS = ("isea3hcellcode", "species", "specieskey", "occurrences")
SPECIES_SEPARATOR = "; "                          # separator used in ``species_list``

## 2. Step-by-step execution

### Step 1: build the query
Shows the SELECT part and the size of the query (the rest is the filters and the three polygons).

In [ ]:
sql = build_sql(ISEA3H_LEVEL)
print(f"{len(sql):,} characters")
print(sql.split("WHERE")[0])

### Step 2: get the GBIF download
If `DOWNLOAD_KEY` is `None`, a new download is requested and the notebook waits for it (this can take a few minutes or longer). Otherwise it reuses that key. Files already on disk are not downloaded again.

In [ ]:
key = DOWNLOAD_KEY or request_download(sql)
cube_csv = download_cube(key, RAW_DIR)
cube_csv

### Step 3: read the GBIF cube
One row per hexagon, species and year-month.

In [ ]:
cube = load_cube(cube_csv)
cube.head()

### Step 4: aggregate per hexagon
Produces one row per hexagon with `n_records`, `richness`, `species_list`, `n_genera`, `n_families` and the richness per kingdom.

In [ ]:
cells = aggregate_cells(cube)
cells.drop(columns="species_list").head()

### Step 5: add the hexagon geometry
Decodes each code, builds the polygon and computes `area_km2`. The layer is in EPSG:4326.

In [ ]:
layer = build_hexagon_layer(cells, EQUAL_AREA_CRS)
levels = sorted(layer["isea3h_level"].unique())
if levels != [ISEA3H_LEVEL]:
    LOGGER.warning("Expected ISEA3H level %s but the cube contains %s", ISEA3H_LEVEL, levels)
print("CRS:", layer.crs)
layer.drop(columns="geometry").head()

### Step 6: keep the hexagons that touch the footprint
Hexagons are kept whole (not clipped). Those outside the footprint are discarded. If `FOOTPRINT` is `None`, every hexagon is kept.

In [ ]:
if FOOTPRINT:
    if not Path(FOOTPRINT).exists():
        raise FileNotFoundError(f"{FOOTPRINT} not found. Run 01_get_footprint.ipynb first.")
    layer = filter_by_footprint(layer, Path(FOOTPRINT))

### Step 7: save the GeoJSON
Writes the hexagon layer to `OUT_DIR` as a single GeoJSON file. It is the only output of the notebook.

In [ ]:
output_path = Path(OUT_DIR) / f"{OUT_NAME}.geojson"
save_geojson(layer, output_path)

The GeoJSON (`<OUT_NAME>.geojson`) is the file to upload to Mapbox. That step is not included in this notebook, see 03_process_vectors.ipynb.